# Stage 2 — metamorphic expansion

Eighteen follow-up cases derived from the eight seeds, two each, three for two seeds (D21), one
relation each from the catalogue its function permits (D22).

A follow-up is a seed with its request **perturbed** and an **expected delta**
stated in the same constraint vocabulary the seed uses. The perturbation and
the delta are chosen by hand and live in `cases/derivations.py`; the shape of
the derived case — what it inherits, which predicates the relation implies,
that it runs at n = 4 and names its base — is `src/metamorphic.py`.

This notebook applies them, shows each follow-up beside its seed, and reports
what a run of all twenty-nine produced.

## What the relations can and cannot claim

Two things were settled before any of these were written, and both change what
is worth asserting.

**`search_places` is capped at five results per call.** So "ask for something
more specific and expect fewer results" — the obvious directional relation for
venue lookup — is not a relation this system has. A tighter constraint makes
the agent search *again*, and `places_surfaced` goes up. What narrowing
actually changes is *which* venues come back, and asking for a second kind of
place changes the shape of the answer outright. The suite asserts those two
and not the count.

**Half the suite has no structural delta, by construction.** D22 gives factual
QA only DIR-1 besides invariance, and advisory invariance alone. For those
four seeds the same tools run, the same nothing is written, and what changed
is the *content* of the answer — which only the judge sees. Their structural
verdict is inherited from the seed and will read `none` whether or not the
agent answered the new question. That is not a defect in the cases; it is
where structural verification stops and scoring starts.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # the subtree, so `src` imports

import json
from collections import Counter

import yaml

from src.cases import CASES_DIR, load_cases, suite_version
from src.metamorphic import ELIGIBLE, RELATION_NAMES, derive
from cases.derivations import DERIVATIONS

## Deriving the eighteen

Regenerated from the seeds every time this runs, so a seed edit cannot leave a stale follow-up behind.

In [2]:
seeds = {c.case_id: c for c in load_cases() if c.provenance == "seeded"}
# Two follow-ups per seed (D21), with two exceptions the suite records
# (D50, D51): `scheduling_move_tram` takes three, because `schedule_block`
# and `unschedule_block` are reachable only under /reschedule and so both
# cases that reach them must derive from that seed; and
# `factual_qa_lisbon_surf_conditions` takes none, because it exists to
# reach `get_climate_normals` and a perturbation of it would buy no tool
# the suite does not already have.
assert set(DERIVATIONS) <= set(seeds)

written = []
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        data = derive(seeds[seed_id], d)
        path = CASES_DIR / f"{data['case_id']}.yaml"
        path.write_text(
            yaml.dump(data, sort_keys=False, allow_unicode=True,
                      default_flow_style=False, width=100),
            encoding="utf-8",
        )
        written.append(data["case_id"])

cases = load_cases()
expanded = [c for c in cases if c.provenance == "metamorphic"]
print(f"{len(written)} follow-ups derived; {len(cases)} cases in the suite")
print(f"executions per run: {sum(c.repetitions_n for c in cases)}  (D21 budget: 128)")
print(f"relations: {dict(sorted(Counter(c.metamorphic_relation for c in expanded).items()))}")
print(f"suite_version: {suite_version()[:16]}…")

17 follow-ups derived; 26 cases in the suite
executions per run: 140  (D21 budget: 128)
relations: {'DIR-1': 5, 'DIR-2': 3, 'DIR-3': 1, 'DIR-4': 2, 'INV-1': 4, 'INV-3': 1, 'INV-4': 1}
suite_version: 1ec4e2a5454069d3…


## Eligibility - why the relations are applied unevenly

The raw counts look arbitrary (five DIR-1, one DIR-3, one INV-2), so the first
thing to say is what the denominator is. A relation is applied only where it
*means* something, and D22 fixes that per function. Two examples make the rule
concrete.

**Eligible - DIR-2 (day changed) on `scheduling_move_tram`.** The seed asks to
move the tram ride to Saturday afternoon; the follow-up asks for Sunday, and
the case's `slot_scheduled` check moves with it. The perturbation names a
different day and the expected outcome *is* a different day, so the follow-up
makes a claim that can fail in one specific, checkable way.

**Not eligible - DIR-2 on `factual_qa_alfama_history`.** The seed asks what
happened in Alfama in the 1755 earthquake. There is no day in the request to
change, and no answer that a changed day would make different. The follow-up's
expected delta would be undefined, and a case that cannot state what ought to
differ is not a test - it is a second copy of the seed wearing a relation's
name.

So the opportunities are uneven before any choice is made: DIR-2, DIR-3 and
DIR-4 are eligible on 4 of the 11 seeds, while INV-1 to INV-4 are eligible on
all 11.

**Within eligibility there is a preference, and it is deliberate.** Directional
relations are used at 25-75% of their opportunity, invariance at 9-36%. A
directional relation asserts the answer *changes*, which fails specifically; an
invariance relation asserts sameness, which is the thing a language model is
most reliably good at. A suite weighted toward invariance would spend its
budget confirming robustness it already has, and report that as coverage.

**The residual limitation, stated rather than dressed up.** INV-2, INV-3 and
INV-4 carry one case each. That is a floor chosen so the catalogue is
instantiated rather than claimed - nothing statistical may be read off a single
case, and the table below says so itself.

In [ ]:
from src import tables

# The grid, not a list per function: the question a reader has is "why does
# THIS relation have so few cases", and that is a column.
tables.eligibility()

In [ ]:
# Use against opportunity, not against the other relations (D61).
# DIR-2/3/4 look starved only until you read their denominator.
tables.relations()

In [ ]:
# The limitation travels with the numbers rather than living only in prose.
from src.metamorphic import relation_report

print(relation_report().split("\n\n", 1)[-1])

## Each follow-up beside its seed

The request, what changed, and the delta the case claims.

In [4]:
def show(seed, follow):
    checks_seed = seed.declared_constraints.checks.model_dump(exclude_none=True)
    checks_follow = follow.declared_constraints.checks.model_dump(exclude_none=True)
    changed = {k: v for k, v in checks_follow.items() if checks_seed.get(k) != v}
    dropped = [k for k in checks_seed if k not in checks_follow]

    print("=" * 100)
    print(f"{follow.case_id}")
    print(f"  {follow.metamorphic_relation} — {RELATION_NAMES[follow.metamorphic_relation]}"
          f"   (base: {seed.case_id}, n={follow.repetitions_n})")
    print(f"  seed asked   : {seed.opening_turn}")
    print(f"  now asks     : {follow.opening_turn}")
    if changed:
        print("  delta claimed:")
        for k, v in changed.items():
            print(f"      {k} = {json.dumps(v, ensure_ascii=False)}")
    else:
        print("  delta claimed: none structurally — the change is in the answer, and the judge scores it")
    if dropped:
        print(f"  dropped      : {dropped}  (the perturbation makes these untrue)")
    if follow.declared_constraints.predicates:
        print(f"  predicates   : {follow.declared_constraints.predicates}")
    print(f"  why          : {follow.notes}")


by_id = {c.case_id: c for c in cases}
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        show(seeds[seed_id], by_id[f"{seed_id}__{d.suffix}"])

venue_lookup_seafood_lisbon__dir1_and_a_hotel
  DIR-1 — constraint added or changed   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good seafood places near Cais do Sodré, nothing too pricey
  now asks     : /find good seafood places near Cais do Sodré, nothing too pricey — and somewhere to stay nearby too
  delta claimed:
      places_surfaced = {"min": 2}
      places_named = {"min": 3}
      places_named_types = ["restaurant", "lodging"]
  why          : The strongest perturbation available to a venue-lookup case: asking for a second kind of place changes the shape of the answer rather than its wording. Chosen over the obvious 'ask for something narrower and expect fewer results', which is not a relation this system has — search_places is capped at five per call, so narrowing pushes the agent to search again and the count rises.
venue_lookup_seafood_lisbon__dir4_not_a_meal
  DIR-4 — negation   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good sea

## What a run of all twenty-nine produced

One repetition each, so every follow-up has its base execution at the same
repetition to compare against (which is what the predicates need).

Not a baseline: `n = 1`, no judge, and the run row says so.

In [5]:
from src.checker import check_run
from src import execution as exmod
from src.db import admin

RUN_ID = None  # set to the expand run's id, or leave None to take the newest

runs = admin().table("study_runs").select("id, notes, started_at").order(
    "started_at", desc=True).limit(10).execute().data
run_id = RUN_ID or next(r["id"] for r in runs if (r["notes"] or "").startswith("EXPAND PROBE"))
print(f"run {run_id[:8]}")

run f62b2c27


In [ ]:
results = check_run(run_id, persist=True)

clean = sum(1 for _, v in results if v.outcome == "none")
print(f"{clean}/{len(results)} executions with no structural finding")

# Sorted so anything with a finding is at the top: a clean run is a long list
# of rows nobody needs to read, and the exceptions are the whole point.
tables.structural(results).sort_values(["findings", "kind"], ascending=[False, True])

## Did the perturbations actually perturb?

A directional relation claims the answer changes. The check above says whether
each execution matched its own constraints; this says whether the follow-up
differed from its seed at all — which is the part a passing verdict can hide.

A follow-up whose venue set is identical to its seed's has not been perturbed,
whatever its checks say.

In [ ]:
# `overlap` is the column to read, and it means opposite things by kind:
# for a DIRECTIONAL relation a high overlap is a warning (the request changed
# and the answer did not), for an INVARIANCE relation it is the expected
# result. `suspect` flags only the first of those.
tables.perturbation(results)

## Review

The cases above are a first set, run end to end. What to look at:

1. **Is the perturbed request what a member would actually type?** The wording
   is the whole input; a stilted one tests a sentence nobody writes.
2. **Is the claimed delta the right claim?** A directional case that asserts
   too much manufactures false regressions (D30); one that asserts too little
   passes whatever the agent does.
3. **The light cases** - factual QA's paraphrases and advisory's - assert
   nothing structurally. Worth deciding whether they earn their place or
   whether the budget is better spent elsewhere.
4. **`INV-2` (typos)** is expected to be the weakest test in the suite, and it
   is deliberately the only one of its relation (D61). It sits on the venue
   lookup because that is the one place a typo reaches past the model into a
   `search_places` query argument rather than being absorbed by it. If it
   passes at 8/8, that is the finding, and the relation is reported as one the
   agent is insensitive to - recording the expectation in advance is what makes
   either outcome interpretable.

Edits go in `cases/derivations.py` and this notebook is re-run; the YAML is
regenerated, never hand-edited.